In [0]:
account_key = dbutils.secrets.get(scope='databricks-scope' ,key = 'databricks-strg-access-key' )

In [0]:
spark.conf.set("fs.azure.account.key.databricksrg2026.dfs.core.windows.net",account_key)

**Ingest _result**

In [0]:
results_df = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
        .json("abfss://raw@databricksrg2026.dfs.core.windows.net/2021-03-21/results.json")

In [0]:
results_cutover = results_df.createOrReplaceTempView("reults_df_cutover")



In [0]:
%sql
select raceId, count(1)
from reults_df_cutover
group by raceId
order by raceId desc

In [0]:
results_df_1 = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
        .json("abfss://raw@databricksrg2026.dfs.core.windows.net/2021-03-28/results.json")

In [0]:
results_rename_df = results_df_1.withColumnRenamed("constructorId","constructor_id")\
.withColumnRenamed("driverId","driver_id")\
    .withColumnRenamed("raceId","race_id")\
    .withColumnRenamed("positionText","position_text")\
    .withColumnRenamed("positionOrder","position_order")\
    .withColumnRenamed("fastestLap","fastest_lap")\
        .withColumnRenamed("fastestLapSpeed","fastest_lap_speed")\
    .withColumnRenamed("fastestLapTime","fastest_lap_time")


In [0]:
result_final=results_rename_df.write.mode("overwrite").parquet("abfss://processed@databricksrg2026.dfs.core.windows.net/2021-03-28/results")

In [0]:
result_final_1=results_rename_df.write.mode("overwrite").save("abfss://processed@databricksrg2026.dfs.core.windows.net/results.json")

In [0]:
spark.sql("""
CREATE TABLE f1_processed.results
USING PARQUET
LOCATION 'abfss://processed@databricksrg2026.dfs.core.windows.net/2021-03-28/results'
""")